In [5]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

In [ ]:
STORE = Path("../store/rwku")
STORE.mkdir(parents=True, exist_ok=True)

In [7]:
# Train: RWKU's refusal training set (200 subjects, ~300 Q/A each).
# Test:  RWKU's official evaluation set (forget_level1/2/3 combined) over the
#        same 200 subjects. Train answers are refusals; test answers are the
#        target facts that the unlearned model should not regurgitate.

tr = load_dataset("jinzhuoran/RWKU", "train_refusal_llama3", split="train")
train_df = pd.DataFrame({
    "concept":  tr["subject"],
    "question": tr["instruction"],
    "answer":   tr["output"],
})

test_parts = []
for lvl in ["forget_level1", "forget_level2", "forget_level3"]:
    te = load_dataset("jinzhuoran/RWKU", lvl, split="test")
    test_parts.append(pd.DataFrame({
        "concept":  te["subject"],
        "question": te["query"],
        "answer":   te["answer"],
    }))
test_df = pd.concat(test_parts, ignore_index=True)

for df in (train_df, test_df):
    for col in ("concept", "question", "answer"):
        df[col] = df[col].fillna("").astype(str).str.strip()
train_df = train_df[(train_df["question"] != "") & (train_df["answer"] != "")].drop_duplicates(subset=["concept", "question"]).reset_index(drop=True)
test_df  = test_df[(test_df["question"]  != "") & (test_df["answer"]  != "")].drop_duplicates(subset=["concept", "question"]).reset_index(drop=True)

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

def stats(df):
    by = df.groupby("concept").size()
    return {"rows": len(df), "concepts": by.shape[0], "rows/concept (mean)": round(by.mean(), 1), "min-max": f"{by.min()}-{by.max()}"}

print(pd.DataFrame({"train": stats(train_df), "test": stats(test_df)}).T)
print(f"\ntrain/test ratio: {len(train_df)/len(test_df):.2f}")

        rows concepts rows/concept (mean) min-max
train  57459      200               287.3  96-300
test   12455      200                62.3   34-80

train/test ratio: 4.61


In [8]:
print("concepts:")
for c in sorted(train_df["concept"].unique()):
    print(" ", c)

concepts:
  50 Cent
  Alanis Morissette
  Alec Baldwin
  Alexander Hamilton
  Anderson Cooper
  Angela Lansbury
  Anna Nicole Smith
  Ariana Grande
  Aristotle
  Barbara Walters
  Betty White
  Beyoncé
  Bill Murray
  Bill Paxton
  Blake Lively
  Bob Barker
  Bob Saget
  Bobby Brown
  Bradley Cooper
  Brendan Fraser
  Brett Favre
  Brooke Shields
  Bruce Lee
  Bruce Springsteen
  Catherine, Princess of Wales
  Charlie Sheen
  Chevy Chase
  Chris Brown
  Christina Aguilera
  Christopher Lloyd
  Chuck Norris
  Cindy Crawford
  Confucius
  Courtney Love
  Dakota Fanning
  Daniel Day-Lewis
  Danny Trejo
  David Crosby
  Demi Moore
  Denise Richards
  Dennis Quaid
  Dionne Warwick
  Don Johnson
  Donald Sutherland
  Donald Trump
  Dr. Dre
  Drew Barrymore
  Dwayne Johnson
  Eddie Murphy
  Elizabeth Hurley
  Elon Musk
  Emilio Estevez
  Evel Knievel
  Faith Hill
  Franklin D. Roosevelt
  Genghis Khan
  Grace Kelly
  Halle Berry
  Harrison Ford
  Henry Kissinger
  Henry Winkler
  Hilary Duff
